# 07 — DQN con capacidad operativa V2

Entrena una Deep Q-Network reproducible implementada con NumPy y compara cinco políticas. Se mantienen únicamente las categorías **bajo, medio y alto**; `3 = Teleorientación` es una acción, no otra categoría.

> Simulación metodológica para priorización operativa; no es una recomendación clínica.


In [ ]:
from pathlib import Path
import os, sys, subprocess
REPO_URL = 'https://github.com/JssSalas/pida-rl-datos-publicos.git'
BRANCH = 'feature/entorno-capacidad-v2'
PROJECT_ROOT = Path('/content/pida-rl-datos-publicos')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'gymnasium', 'pandas'])
if not PROJECT_ROOT.exists():
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(PROJECT_ROOT)])
else:
    subprocess.check_call(['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', BRANCH])
    subprocess.check_call(['git', '-C', str(PROJECT_ROOT), 'checkout', BRANCH])
    subprocess.check_call(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only', 'origin', BRANCH])
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


## Configuración

Se usan semillas separadas para entrenamiento y evaluación. La red objetivo, el replay buffer y todos los hiperparámetros quedan explícitos para asegurar trazabilidad.


In [ ]:
CAPACIDAD_MENSUAL = 100
COSTOS_ACCION = {0: 0, 1: 1, 2: 3, 3: 5}
HORIZONTE_MESES = 12
PASOS_ENTRENAMIENTO = 100_000
SEMILLA_ENTRENAMIENTO = 20_000
SEMILLAS_EVALUACION = (42, 123, 2026)
CAPAS_OCULTAS = (64, 64)
GAMMA = 0.95
LEARNING_RATE = 1e-3
EPSILON_INICIAL = 1.0
EPSILON_MIN = 0.05
EPSILON_DECAY = 0.99995
REPLAY_CAPACITY = 50_000
WARMUP_STEPS = 1_000
BATCH_SIZE = 64
TRAIN_FREQUENCY = 4
TARGET_UPDATE_FREQUENCY = 500
LOG_INTERVAL = 1_000


## Cohorte procesada

Se reutiliza el archivo público procesado por el notebook 03; este notebook no fabrica pacientes.


In [ ]:
from google.colab import drive
import pandas as pd
drive.mount('/content/drive', force_remount=False)
ROOT = Path('/content/drive/MyDrive/PIDA-RL-Diabetes')
COHORT_PATH = ROOT / 'data' / 'processed' / 'cohorte_diabetes.csv'
OUTPUT_DIR = ROOT / 'results' / 'capacidad_v2' / 'dqn'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cohorte = pd.read_csv(COHORT_PATH, dtype={'FOLIO_INT': 'string'})
assert set(cohorte['categoria_riesgo'].unique()) <= {'bajo', 'medio', 'alto'}
print(cohorte.shape, cohorte['categoria_riesgo'].value_counts().to_dict())


## Entrenamiento DQN

La red recibe las 13 variables normalizadas. El agente utiliza replay buffer, una red objetivo y actualización de Bellman con Adam y recorte de gradiente.


In [ ]:
from src.environment.diabetes_followup_env import DiabetesFollowUpEnv
from src.policies.dqn import entrenar_dqn

def crear_entorno():
    return DiabetesFollowUpEnv(cohorte, CAPACIDAD_MENSUAL, COSTOS_ACCION, HORIZONTE_MESES)

agente_dqn, historial_dqn = entrenar_dqn(
    crear_entorno, PASOS_ENTRENAMIENTO, SEMILLA_ENTRENAMIENTO, CAPAS_OCULTAS,
    GAMMA, LEARNING_RATE, EPSILON_INICIAL, EPSILON_MIN, EPSILON_DECAY,
    REPLAY_CAPACITY, WARMUP_STEPS, BATCH_SIZE, TRAIN_FREQUENCY,
    TARGET_UPDATE_FREQUENCY, LOG_INTERVAL,
)
historial_dqn.to_csv(OUTPUT_DIR / 'historial_entrenamiento_dqn.csv', index=False)
agente_dqn.guardar(OUTPUT_DIR / 'dqn_v2.npz')
display(historial_dqn.tail())


## Evaluación común

DQN se ejecuta sin exploración y se compara con las políticas base y Q-learning usando exactamente las mismas semillas y restricciones. Para incluir Q-learning, se carga el modelo producido por el notebook 06.


In [ ]:
from src.evaluation.evaluate_policies import evaluar_politicas
from src.evaluation.reporting import construir_reporte, exportar_reporte_csv
from src.policies.baselines import POLITICAS_BASE
from src.policies.dqn import crear_politica_dqn
from src.policies.q_learning import QLearningAgent, crear_politica_q_learning

Q_PATH = ROOT / 'results' / 'capacidad_v2' / 'q_learning' / 'q_learning_v2.npz'
politicas = dict(POLITICAS_BASE)
if Q_PATH.exists():
    agente_q = QLearningAgent.cargar(Q_PATH, seed=SEMILLA_ENTRENAMIENTO)
    politicas['Q-learning'] = crear_politica_q_learning(agente_q)
else:
    print('Aviso: ejecute el notebook 06 para incluir Q-learning.')
politicas['DQN'] = crear_politica_dqn(agente_dqn)
resultados = evaluar_politicas(crear_entorno, politicas, SEMILLAS_EVALUACION)
reporte = construir_reporte(resultados)
exportar_reporte_csv(reporte, OUTPUT_DIR)
detalle = reporte['detalle']
assert set(detalle['categoria_riesgo']) <= {'bajo', 'medio', 'alto'}
assert (detalle['recursos_despues'] >= 0).all()
assert (detalle['recursos_usados_mes'] <= detalle['capacidad_mensual']).all()
display(reporte['resumen_algoritmos'].round(4))
display(reporte['clasificaciones_propuestas'].round(2))
display(reporte['clasificaciones_ejecutadas'].round(2))


## Interpretación responsable

- Evalúe recompensa, eventos, cobertura, recursos y ajustes por capacidad de manera conjunta.
- Diferencie siempre acciones propuestas de acciones ejecutadas.
- Revise estabilidad entre semillas y la curva de pérdida antes de interpretar DQN.
- La red aprende la recompensa del simulador; no demuestra efectividad clínica real.
- Si cambia la recompensa, la cohorte, los costos o la capacidad, Q-learning y DQN deben reentrenarse.
